## NISAR Data Access

This notebook converts sample ground truth data coordinates (latitude and longitude) into corresponding NISAR raster pixel indices. Identifying the correct pixel location for each site allows direct extraction of NISAR data values at field observation points.

The workflow includes transforming geographic coordinates into the NISAR product’s coordinate system, locating the nearest raster pixel, and storing the associated row and column indices for analysis. The resulting site locations and pixel footprints are also visualized on an interactive Folium map to confirm alignment between ground sites and the NISAR raster grid.

## Notebook Overview

1. [Prerequisites](access-prereqs)
1. [Selecting a NISAR Procdct](access-selectprod)
2. [Download a NISAR HDF5 File](access-doownload)
2. [Open the file in H5Py](access-h5py)
3. [Opening the NISAR file in xarray](access-xarray)
4. [Opening the NISAR file in ISCE3](access-isce3)
9. [Summary](access-summary)
10. [Resources and References](access-resources)

<hr>

(access-prereqs)=
## 1. Prerequisites
| Prerequisite | Importance | Notes |
| --- | --- | --- |
| [The software environment for this cookbook must be installed](https://github.com/ASFOpenSARlab/NISAR_GCOV_Cookbook/blob/main/notebooks/create_software_environment.ipynb) | Necessary | |

- **Rough Notebook Time Estimate**: 10 minutes

<hr>

## 2. Comparing Methods for Accessing NISAR Data

The three approaches shown in this notebook provide different levels of access and flexibility when working with NISAR HDF5 files.

| Method   | Best For | Pros | Cons |
|----------|----------|------|------|
| **h5py** | Direct file access | Minimal dependencies, full control over HDF5 structure | Requires long paths, manual navigation of groups and datasets |
| **xarray** | Data analysis and visualization | Easy to work with arrays, labeled dimensions, cleaner syntax | May require specifying groups, slightly more setup |
| **ISCE3** | SAR processing and NISAR-specific workflows | Designed for NISAR data, understands radar geometry and metadata | Does not easily pull all types of metadata, requires additional installation, may not work in all environments |

---

### When should you use each?

- Use **h5py** when you want to explore the file structure or access raw data directly.
- Use **xarray** when you want a cleaner, more user-friendly way to analyze and visualize datasets.
- Use **ISCE3** when working on advanced SAR and NISAR-specific applications.

(access-selectprod)=
## 3. Selecting a NISAR Product

NISAR products are identified by a four-letter abbreviation. In the download example below, the product type is controlled by the `processingLevel` field. To learn more about invidicual products, see the [NISAR Data User Guide](https://nisar-docs.asf.alaska.edu/).

For this walkthrough, we use:

- **GCOV** — Geocoded Polarimetric Covariance Matrix

Other NISAR products include:

### L0 Products
- **RRSD** — Radar Raw Signal Data

### L1 Products
- **RSLC** — Range-Doppler Single Look Complex
- **RIFG** — Range-Doppler Interferogram
- **ROFF** — Range-Doppler Pixel Offsets
- **RUNW** — Range-Doppler Unwrapped Interferogram

### L2 Products
- **GSLC** — Geocoded Single Look Complex
- **GOFF** — Geocoded Pixel Offsets
- **GCOV** — Geocoded Polarimetric Covariance Matrix
- **GUNW** — Geocoded Unwrapped Interferogram

### L3 Products
- **SME2** — Soil Moisture

#### To search for a different NISAR product, replace `"GCOV"` in the `processingLevel` field with the four-letter abbreviation for your product of interest.

<hr>

(access-download)=

## 4. Download a NISAR HDF5 file

### 4a. Search for NISAR data. For this examlpe, we will use a GCOV data.

In [19]:
import os
import asf_search as asf
from datetime import datetime
from getpass import getpass
from pathlib import Path

# authenticate with Earthdata Login
session = asf.ASFSession()
session.auth_with_creds(input('EDL Username'), getpass('EDL Password'))

# insert start and end dates. Note that dates and times are inclusive. 
start_date = datetime(2025, 12, 4)
end_date = datetime(2025, 12, 5)
area_of_interest = "POLYGON((40.9131 12.3904,41.8891 12.3904,41.8891 13.2454,40.9131 13.2454,40.9131 12.3904))" 

opts = asf.ASFSearchOptions(
    **{
        "maxResults": 250,                    # limit the number of returned results
        "intersectsWith": area_of_interest,   # spatial filter (WKT geometry)
        "start": start_date,                  # temporal filter (start)
        "end": end_date,                      # temporal filter (end)
        
        # to switch NISAR products, change the four letter abbreviation next to processingLevel
        "processingLevel": ["GCOV"],          # product type / processing level
        "dataset": ["NISAR"],                 # mission/dataset name
        "productionConfiguration": ["PR"],    # production config (e.g., provisional)
        "session": session,                   # use authenticated session from above
    }
)

response = asf.search(opts=opts)

pattern = r'^(?!.*QA_STATS).*'

h5_files = response.find_urls(extension='.h5', pattern=pattern, directAccess=False)
h5_files

EDL Username jwhite124
EDL Password ········


['https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05007_N_F_J_001/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05007_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05009_N_F_J_001/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05009_N_F_J_001.h5']

### 4b. Download the data

In [21]:
data_dir = Path.home() / "NISAR_data_access_example"
data_dir.mkdir(exist_ok=True)
print(f'data_dir: {data_dir}')

#download files
asf.download_urls(h5_files, data_dir, session=session, processes=4)

data_dir: /home/jovyan/NISAR_data_access_example


<hr>

(access-h5py)=
## 5. Opening NISAR Data with h5py

The `h5py` library allows you to access HDF5 files directly in Python. NISAR products are stored as hierarchical datasets, similar to folders and files.

In this example, we open a NISAR file, navigate to the metadata group, read the `incidenceAngle` dataset and print the shape.

In [22]:
import h5py

h5_h5py = list(data_dir.glob("NISAR_L2_*_GCOV*.h5"))[0]

with h5py.File(h5_h5py, "r") as f:
    incidence = f["/science/LSAR/GCOV/metadata/radarGrid/incidenceAngle"][:]

print("incidenceAngle shape:", incidence.shape)

incidenceAngle shape: (21, 680, 695)


<hr>

(access-xarray)=
## 6. Opening NISAR Data with xarray

The `xarray` library provides a convenient way to work with labeled arrays. Because NISAR HDF5 files are hierarchical, we use `datatree` to access groups within the file.

In this example, we open a NISAR file, access the metadata group, and read the same `incidenceAngle` dataset.

In [23]:
#ignore warnings
import warnings
warnings.filterwarnings("ignore")

import xarray as xr

h5_xarray = list(data_dir.glob("NISAR_L2_*_GCOV*.h5"))[0]

ds = xr.open_dataset(
    h5_xarray,
    engine="h5netcdf",
    group="/science/LSAR/GCOV/metadata/radarGrid"
)

incidence = ds["incidenceAngle"].data
print("incidenceAngle shape:", incidence.shape)

ds.close()

incidenceAngle shape: (21, 680, 695)


<hr>

(access-isce3)=
## 7. ISCE3

The NISAR product reader (part of ISCE3) provides a more complex interface for working with NISAR data. It understands the structure of NISAR products and includes built-in attributes and methods for accessing product information and image datasets.

Unlike `h5py` and `xarray`, which access datasets directly through the HDF5 structure, `ISCE3` interacts with the data through class methods and member variables.

## 7a. Opening NISAR Metadata with ISCE3

In this example, we first access general product information, `polarizations` metadata, using reader member variables.

These values come from the product metadata, but are accessed differently than in the previous examples. Instead of navigating the file structure (as with `h5py` or `xarray`), `ISCE3` exposes selected metadata through object attributes.

While some metadata can be accessed this way, many metadata datasets (such as `incidenceAngle`) are not available through a simple or consistent interface in ISCE3. For this reason, `h5py` or `xarray` may be more ideal for accessing metadata.

In [24]:
from nisar.products.readers import open_product

h5_isce3 = list(data_dir.glob("NISAR_L2_*.h5"))[0]

product = open_product(h5_isce3)

print("polarizations:", product.polarizations)

polarizations: {'A': ['HH', 'HV'], 'B': ['HH', 'HV']}


## 7b. Using ISCE3 to read image data 

Next, we use the `ISCE3` reader to access an image dataset using the `getImageDataset()` method.

This method provides a convenient way to access image layers (such as polarization datasets), which are central to many SAR and NISAR workflows. This type of access is a more common use of ISCE3, as it is designed to work with image data and SAR-specific processing rather than general metadata retrieval.

In [25]:
from nisar.products.readers import open_product

h5_hvhv = list(data_dir.glob("NISAR_L2_*_GCOV*.h5"))[0]

hvhv_data = open_product(h5_hvhv)

hvhv = hvhv_data.getImageDataset(frequency="A", polarization="HVHV")

print("HVHV shape:", hvhv[:, :].shape)

HVHV shape: (16704, 17064)


<hr>

(access-summary)=
## 8. Summary

You have now three methods and examples for opening NISAR data.
<hr>

(access-resources)=
## 9. Resources and references
- [HDF5 Documentation](https://support.hdfgroup.org/documentation/)
- [h5py](https://github.com/h5py/h5py)
- [xarray](https://docs.xarray.dev/en/stable/)
- [isce3](https://github.com/isce-framework/isce3)
- [NISAR Data User Guide](https://nisar-docs.asf.alaska.edu/)

**Author:** Julia White